## Deliverable 2

task 1:

In [ ]:
from pathlib import Path 

import re

def read_file(file_path):

    path = Path(file_path)

    try: 
        file_text_content = path.read_text(encoding="utf-8")
        return(file_text_content, 'utf-8')

    except UnicodeDecodeError:
        file_text_content = path.read_text(encoding="iso-8859-1")
        return(file_text_content, 'iso-8859-1') 

def parse_log(log_text):
    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})] "
        r"\[(?P<register_id>[^\]]+)\] "
        r"item=(?P<item>\w+) "
        r"price=(?P<price>\d+\.\d+) "
        r"(?P<currency>[A-Z]{3}) "
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\) "
        )
    records = []
    for line in log_text.splitlines():
        match = pattern.search(line)

        if match:
            record = match.groupdict()

            record["price"] = float(record["price"])

            records.append(record)
    return records 


Task 2: 


In [ ]:
import json

def find_log_files(dir_path): 
    dir_path = Path(dir_path)
    log_files = []


    for item in dir_path.iterdir():
        if item.is_dir(): 
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log": 
            log_files.append(item)
    return log_files

def find_active_registers(data):
    active_registers = set()

    if isinstance(data, dict):
        if "id" in data and data.get("active") is True:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))


    elif isinstance(data, list):

        for item in data: 
            active_registers.update(find_active_registers(item))

    return active_registers

with open("data/store_config.json", "r", encoding="utf-8") as f: # not sure if this is needed? 
    metadata = json.load(f)

active_registers_in_metadata = find_active_registers(metadata)
        

Task 3: 

In [ ]:
import numpy as np


def removing_price_outliers(data): 
    
    price = np.array(data, dtype=float).ravel()
    mean = np.mean(price)
    std = np.std(price)

    outliers = np.abs(price - mean) > 3 * std

    valid_prices = price[~outliers]
    valid_median = np.median(valid_prices)

    price[outliers] = valid_median 

    return price




